In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import glob
from scipy.stats import spearmanr
from statsmodels.stats.multitest import multipletests
import os

# TEST phenotypes

In [2]:
dpheno = pd.read_csv("../44_train_and_test/00_combine_samples_means.tsv", sep="\t", header=0)
dpheno["POP_ID"] = dpheno["POP_ID"].astype(str).astype(object)
#dpheno = pd.merge(dpheno_, dmeta, on = "POP_ID", how = "left")
dpheno = dpheno[dpheno['SET'] == 'TEST'].reset_index(drop=True)
#dpheno = dpheno[~dpheno['group_cluster'].isin(['ME','WI'])].reset_index(drop=True)
print(dpheno.shape)
dpheno["sample"] = dpheno["POP_SITE"]
dpheno.head()

print(dpheno["sample"].drop_duplicates().shape)

(2048, 8)
(143,)


### Filtering out rows (pop_site x trait ) which have means obtained from less than 3 samples

In [3]:
dpheno[dpheno["n"] < 3].shape
#dpheno[dpheno["n"] < 3]

(48, 9)

In [4]:
dpheno = dpheno[dpheno['n']>2].reset_index(drop=True)
print(dpheno.shape)

(2000, 9)


# TRAIN phenotypes

In [5]:
tpheno = pd.read_csv("../44_train_and_test/00_combine_samples_means.tsv", sep="\t", header=0)
tpheno["POP_ID"] = tpheno["POP_ID"].astype(str).astype(object)
#tpheno = pd.merge(tpheno_, dmeta, on = "POP_ID", how = "left")
tpheno = tpheno[tpheno['SET'] == 'TRAIN'].reset_index(drop=True)
#tpheno = tpheno[~tpheno['group_cluster'].isin(['ME','WI'])].reset_index(drop=True)
print(tpheno.shape)
tpheno["sample"] = tpheno["POP_SITE"]

### Filtering out rows (pop_site x trait ) which have means obtained from less than 3 samples
print(tpheno[tpheno["n"] < 3].shape)

tpheno = tpheno[tpheno['n']>2].reset_index(drop=True)
print(tpheno.shape)
tpheno.head()

print(tpheno["sample"].drop_duplicates().shape)

(1738, 8)
(108, 9)
(1630, 9)
(115,)


# Combining genetic offset sets

In [6]:
traits = ["Height", "Biomass_Increment", "Biomass_Increment_1980", "Biomass_Increment_1985", 
                "Biomass_Increment_1990", "Biomass_Increment_1995", "Biomass_Increment_2000", 
                "Biomass_Increment_2005", "Biomass_Increment_2010", "Biomass_Increment_2015",
                "Average_Ring_Density", "DBH", "Rs", "Rl", "Rr","Rc"]

gardens = ["PR","ML","CH","AC"]
groups = ["East","Central","West"]

#dsets = ["100","1000","lfmm","RDA","RDAcorrected", "10000","all"]
#dsets = ["100","1000","10000","lfmm","RDA","RDAcorrected"]
dsets = ["1000var5"]
dsets

['1000var5']

In [7]:
D = []

for trait in traits:
    for garden in gardens:
        for group in groups:
            file_path = "../958B_garden_offset_run_clusters_var5/results/01_run_gradient_forest_" + garden + "_" + group + "_" + trait + ".tsv"
            if os.path.exists(file_path):
                df_ = pd.read_csv(file_path, sep="\t", header=0)
                df_["garden"] = garden
                df_["marker"] = dsets[0]
                df_["trait"] = trait
                df_["train_group"] = group
                #df_["garden"] = garden
                D.append(df_)
dD = pd.concat(D)
print(dD.shape)
dD.head()

(3959, 18)


,Trait_name,sample,SITE_ID,group,dPP,CMI,fallMT,now_dPP,now_CMI,now_fallMT,new_dPP,new_CMI,new_fallMT,offset,garden,marker,trait,train_group
0,Height,2209_PR,PR,West,56.200000,14.112083,-5.880183,0.120235,0.022183,-0.024831,0.089448,0.012289,0.01055,0.047933,PR,1000var5,Height,Central
1,Height,325_PR,PR,Central,46.066667,59.986610,1.989890,0.030898,0.067959,0.011693,0.089448,0.012289,0.01055,0.080800,PR,1000var5,Height,Central
2,Height,4351_PR,PR,Central,40.166667,38.597673,7.302088,-0.004206,0.051117,0.035990,0.089448,0.012289,0.01055,0.104527,PR,1000var5,Height,Central
3,Height,4420_PR,PR,West,60.133333,-1.820773,-4.420879,0.150764,-0.002057,-0.018098,0.089448,0.012289,0.01055,0.069182,PR,1000var5,Height,Central
4,Height,6805_PR,PR,East,49.466667,66.059520,2.622601,0.065592,0.074839,0.014448,0.089448,0.012289,0.01055,0.067058,PR,1000var5,Height,Central


In [8]:
# skipping group column here because it's incomplete for all pops
dD_sub = dD[["sample","SITE_ID","garden","offset","marker","trait","train_group"]].reset_index(drop=True)
print(dD_sub.shape)
dD_sub.head()
print(dD_sub["sample"].drop_duplicates().shape)

(3959, 7)
(139,)


In [9]:
#dD_sub['group'].drop_duplicates()
dD_sub['garden'].drop_duplicates()

0      PR
52     ML
136    CH
247    AC
Name: garden, dtype: object

### Test

In [10]:
# phenotypes come with the group column (complete)
dD_merge = pd.merge(dD_sub, dpheno, left_on = ["sample","SITE_ID","trait"], right_on = ["sample","SITE_ID","Trait_name"], how = "left")
print(dD_merge.shape)
dD_merge.head()

# There are rows with missing information - these are generally pop_sites x traits where phenotypes were measured on less than 3 samples  
# Removing these
dD_merge.loc[dD_merge["SET"].isna(),['sample']].drop_duplicates().shape
dD_merge.loc[dD_merge["SET"].isna(),['sample']].drop_duplicates()

dD_merge = dD_merge.dropna().reset_index(drop=True)
print(dD_merge.shape)

dD_merge[dD_merge["SET"].isna()]

(3959, 14)
(3880, 14)


,sample,SITE_ID,garden,offset,marker,trait,train_group,Trait_name,POP_SITE,POP_ID,SET,mean,sd,n


#### Adding groups

In [11]:
dmeta = pd.read_csv("../METADATA/populations_genetic_groups.tsv", sep="\t", header=0)
dmeta.head()

tr_merge = pd.merge(dD_merge, dmeta, on = "POP_ID", how = "left")
tr_merge[tr_merge["group"].isna()]
tr_merge.head()

,sample,SITE_ID,garden,offset,marker,trait,train_group,Trait_name,POP_SITE,POP_ID,SET,mean,sd,n,group
0,2209_PR,PR,PR,0.047933,1000var5,Height,Central,Height,2209_PR,2209,TEST,980.525258,302.158899,4.0,West
1,325_PR,PR,PR,0.080800,1000var5,Height,Central,Height,325_PR,325,TEST,1239.891145,121.743290,8.0,Central
2,4351_PR,PR,PR,0.104527,1000var5,Height,Central,Height,4351_PR,4351,TEST,1423.598605,68.068593,4.0,Central
3,4420_PR,PR,PR,0.069182,1000var5,Height,Central,Height,4420_PR,4420,TEST,678.153897,154.272486,7.0,West
4,6805_PR,PR,PR,0.067058,1000var5,Height,Central,Height,6805_PR,6805,TEST,1353.974749,229.855027,7.0,East


### Train

In [12]:
tD_merge = pd.merge(dD_sub, tpheno, left_on = ["sample","SITE_ID","trait"], right_on = ["sample","SITE_ID","Trait_name"], how = "left")

# Dropping pops not present in TRAIN (nan)
tD_merge = tD_merge.dropna().reset_index(drop=True)

print(tD_merge.shape)
tD_merge.tail()
#tD_merge[tD_merge["SET"].isna()]

ts_merge = pd.merge(tD_merge, dmeta, on = "POP_ID", how = "left")
ts_merge[ts_merge["group"].isna()]
ts_merge.head()

### Removing samples absent from the train (=removing cluster removed from the training)
# For the trining phenotype set, I only keep the samples on which the model was trained on (pop_sites from the same cluster) 
ts_merge = ts_merge[ts_merge["train_group"] == ts_merge["group"]].dropna().reset_index(drop=True)
print(ts_merge.shape)
ts_merge.head()

(3565, 14)
(1447, 15)


,sample,SITE_ID,garden,offset,marker,trait,train_group,Trait_name,POP_SITE,POP_ID,SET,mean,sd,n,group
0,325_PR,PR,PR,0.080800,1000var5,Height,Central,Height,325_PR,325,TRAIN,1202.391145,197.315314,7.0,Central
1,4351_PR,PR,PR,0.104527,1000var5,Height,Central,Height,4351_PR,4351,TRAIN,1311.931938,178.648122,12.0,Central
2,6905_PR,PR,PR,0.099729,1000var5,Height,Central,Height,6905_PR,6905,TRAIN,1405.322260,192.242407,7.0,Central
3,6907_PR,PR,PR,0.088781,1000var5,Height,Central,Height,6907_PR,6907,TRAIN,1579.492102,100.344860,11.0,Central
4,6916_PR,PR,PR,0.068075,1000var5,Height,Central,Height,6916_PR,6916,TRAIN,1517.807568,129.590281,13.0,Central


In [13]:
#dcheck = ts_merge[(ts_merge['trait'] == "Average_Ring_Density") & (ts_merge['group'] == "Central") & (ts_merge['SITE_ID'] == "AC")]
#dcheck

In [14]:
#spearmanr(dcheck['offset'], dcheck['mean'], nan_policy='omit')

### Combining

In [15]:
df_combo = pd.concat([tr_merge, ts_merge], ignore_index=True)
df_combo.head()
print(df_combo.shape)
df_combo[df_combo['SET'].isna()]
df_combo[df_combo['POP_ID'].isna()]

(5327, 15)


,sample,SITE_ID,garden,offset,marker,trait,train_group,Trait_name,POP_SITE,POP_ID,SET,mean,sd,n,group


In [16]:
df_combo["POP_ID"] = df_combo["POP_ID"].astype(int).astype(str)
df_combo["POP_ID"]
df_combo.loc[df_combo["group"].isna(),'sample'].drop_duplicates()

Series([], Name: sample, dtype: object)

### Removing WI and ME

In [17]:
dc_combo = df_combo[~df_combo['group'].isin(["WI","ME"])].reset_index(drop=True)
print(dc_combo.shape)

(5199, 15)


In [18]:
dc_combo.to_csv("08_garden_clusters.tsv", sep = "\t", header=True, index=False)

# END